# Notebook 09 — Train XGBoost Models

## Goal
Train three XGBoost regressors with different feature sets and compare them:

1. **Macro-only** — 8 macroeconomic lag features
2. **News-only** — 7 news count features
3. **Macro + News** — all 15 features

**Main research question answered here:**  
Does adding news features to macro features improve payroll forecasts?

---

## What can go wrong
- XGBoost may overfit on the small training set — check val vs. test performance
- News-only model may perform poorly (counts may be too noisy)
- If macro+news does not beat macro-only, do NOT claim news improves forecasting
- COVID outliers will distort test metrics — inspect crisis vs. non-crisis separately

---

## What you must inspect
- Feature lists for each model
- Val and test metrics side by side
- Does macro+news beat macro-only? (honest reporting required)
- Are any predictions extreme outliers?

In [ ]:
import pathlib, json, datetime, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

MODEL_READY_PATH = DRIVE_ROOT / 'data' / 'model_ready' / 'labor_forecasting_dataset_approved.parquet'
MODELS_DIR = DRIVE_ROOT / 'artifacts' / 'models'
PREDS_DIR = DRIVE_ROOT / 'outputs' / 'predictions'
METRICS_DIR = DRIVE_ROOT / 'outputs' / 'metrics'

ap_path = DRIVE_ROOT / 'approvals' / '07_model_ready_dataset_approved.json'
if not ap_path.exists():
    raise FileNotFoundError('STOP: Notebook 07 not approved.')
with open(ap_path) as f:
    ap = json.load(f)
assert ap['approved'], 'NB 07 not approved.'

with open(REPO_DIR / 'configs' / 'base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open(REPO_DIR / 'configs' / 'model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

_splits_key = 'test_splits' if base_cfg.get('test_mode', False) else 'splits'
_splits = model_cfg[_splits_key]
TRAIN_END = pd.Timestamp(_splits['train_end'])
VAL_END = pd.Timestamp(_splits['validation_end'])
TEST_START = pd.Timestamp(_splits['test_start'])
TARGET = model_cfg['target_column']
SEED = 42

MACRO_FEATURES = model_cfg['macro_features']
NEWS_FEATURES = model_cfg['news_features']
ALL_FEATURES = MACRO_FEATURES + NEWS_FEATURES

MODE = 'TEST (quick run)' if base_cfg.get('test_mode', False) else 'FULL (research run)'
print(f'Mode   : {MODE}')
print(f'Splits : train≤{_splits["train_end"]} | val≤{_splits["validation_end"]} | test≥{_splits["test_start"]}')

model_df = pd.read_parquet(MODEL_READY_PATH)
model_df['forecast_date'] = pd.to_datetime(model_df['forecast_date'])
model_df = model_df.dropna(subset=[TARGET]).sort_values('forecast_date').reset_index(drop=True)

train_df = model_df[model_df['forecast_date'] <= TRAIN_END]
val_df = model_df[(model_df['forecast_date'] > TRAIN_END) & (model_df['forecast_date'] <= VAL_END)]
test_df = model_df[model_df['forecast_date'] >= TEST_START]

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Macro features ({len(MACRO_FEATURES)}): {MACRO_FEATURES}')
print(f'News features ({len(NEWS_FEATURES)}): {NEWS_FEATURES}')

In [ ]:
def compute_metrics(actuals, predictions, model_name, period):
    actuals = np.array(actuals)
    predictions = np.array(predictions)
    mask = ~(np.isnan(actuals) | np.isnan(predictions))
    a, p = actuals[mask], predictions[mask]
    return {
        'model': model_name, 'period': period, 'n': int(mask.sum()),
        'MAE': round(float(mean_absolute_error(a, p)), 2),
        'RMSE': round(float(np.sqrt(mean_squared_error(a, p))), 2),
        'DA': round(float(np.mean(np.sign(a) == np.sign(p))), 4),
    }

def train_xgb(train_df, features, target, params):
    X_train = train_df[features].fillna(0)
    y_train = train_df[target]
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, verbose=False)
    return model

def predict_xgb(model, df, features):
    X = df[features].fillna(0)
    return model.predict(X)

def show_model_results(model_name, features, train_df, val_df, test_df, val_pred, test_pred, target):
    print(f'\n{"="*60}')
    print(f'MODEL: {model_name}')
    print(f'{"="*60}')
    print(f'Features ({len(features)}): {features}')
    print(f'Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}')
    m_val = compute_metrics(val_df[target], val_pred, model_name, 'val')
    m_test = compute_metrics(test_df[target], test_pred, model_name, 'test')
    print(f'Val  — MAE={m_val["MAE"]:.1f}K, RMSE={m_val["RMSE"]:.1f}K, DA={m_val["DA"]:.2%}')
    print(f'Test — MAE={m_test["MAE"]:.1f}K, RMSE={m_test["RMSE"]:.1f}K, DA={m_test["DA"]:.2%}')
    sample = pd.DataFrame({
        'forecast_date': test_df['forecast_date'].values[-10:],
        'actual': test_df[target].values[-10:],
        'prediction': test_pred[-10:],
        'error': test_df[target].values[-10:] - test_pred[-10:],
    })
    print(sample.to_string(index=False))
    return m_val, m_test

all_metrics = []
print('Helper functions defined.')

## Bayesian Hyperparameter Search

Uses **Optuna TPE sampler** to find the best XGBoost hyperparameters for each model variant.

- Optimizes **validation MAE only** — test set is never touched during search
- Runs 3 independent studies (one per feature set)
- Number of trials: controlled by `param_search` in `model_config.yaml` (`n_trials_test` in test mode, `n_trials` in full mode)
- Best params are saved to Drive and used for final model training below

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

ps_cfg = model_cfg.get('param_search', {})
N_TRIALS = ps_cfg.get('n_trials_test', 30) if base_cfg.get('test_mode', False) else ps_cfg.get('n_trials', 100)
ss = ps_cfg.get('search_space', {})

print(f'Bayesian search — {N_TRIALS} trials per model (mode: {MODE})')
print('Metric: validation MAE — test set is never used during search')

def make_objective(feature_list):
    available = [f for f in feature_list if f in train_df.columns]
    X_train = train_df[available].fillna(0).values
    y_train = train_df[TARGET].values
    X_val = val_df[available].fillna(0).values
    y_val = val_df[TARGET].values

    def objective(trial):
        params = {
            'n_estimators':    trial.suggest_int(  'n_estimators',    *ss.get('n_estimators',    [100, 800])),
            'max_depth':       trial.suggest_int(  'max_depth',       *ss.get('max_depth',       [3, 8])),
            'learning_rate':   trial.suggest_float('learning_rate',   *ss.get('learning_rate',   [0.01, 0.3]), log=True),
            'subsample':       trial.suggest_float('subsample',       *ss.get('subsample',       [0.6, 1.0])),
            'colsample_bytree':trial.suggest_float('colsample_bytree',*ss.get('colsample_bytree',[0.6, 1.0])),
            'min_child_weight':trial.suggest_int(  'min_child_weight',*ss.get('min_child_weight',[1, 10])),
            'reg_alpha':       trial.suggest_float('reg_alpha',       *ss.get('reg_alpha',       [0.0, 1.0])),
            'reg_lambda':      trial.suggest_float('reg_lambda',      *ss.get('reg_lambda',      [1.0, 10.0])),
            'random_state': SEED, 'n_jobs': -1, 'verbosity': 0,
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        return mean_absolute_error(y_val, model.predict(X_val))

    return objective

study_macro = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_macro.optimize(make_objective(MACRO_FEATURES), n_trials=N_TRIALS, show_progress_bar=True)

study_news = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_news.optimize(make_objective(NEWS_FEATURES), n_trials=N_TRIALS, show_progress_bar=True)

study_mn = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_mn.optimize(make_objective(ALL_FEATURES), n_trials=N_TRIALS, show_progress_bar=True)

best_params_macro = {**study_macro.best_params, 'random_state': SEED, 'n_jobs': -1, 'verbosity': 0}
best_params_news  = {**study_news.best_params,  'random_state': SEED, 'n_jobs': -1, 'verbosity': 0}
best_params_mn    = {**study_mn.best_params,    'random_state': SEED, 'n_jobs': -1, 'verbosity': 0}

print(f'\nBest val MAE:')
print(f'  macro-only : {study_macro.best_value:.2f}K  — {study_macro.best_params}')
print(f'  news-only  : {study_news.best_value:.2f}K  — {study_news.best_params}')
print(f'  macro+news : {study_mn.best_value:.2f}K  — {study_mn.best_params}')

best_params_path = METRICS_DIR / 'best_hyperparams.json'
with open(best_params_path, 'w') as f:
    json.dump({
        'xgboost_macro_only': study_macro.best_params,
        'xgboost_news_only':  study_news.best_params,
        'xgboost_macro_news': study_mn.best_params,
        'n_trials': N_TRIALS,
        'optimized_metric': 'val_mae',
    }, f, indent=2)
print(f'Best params saved: {best_params_path}')

## Model 1 — XGBoost Macro-Only

In [ ]:
available_macro = [f for f in MACRO_FEATURES if f in train_df.columns]
missing_macro = [f for f in MACRO_FEATURES if f not in train_df.columns]
if missing_macro:
    print(f'WARNING: Missing macro features: {missing_macro}')

xgb_macro = train_xgb(train_df, available_macro, TARGET, best_params_macro)

macro_val_pred = predict_xgb(xgb_macro, val_df, available_macro)
macro_test_pred = predict_xgb(xgb_macro, test_df, available_macro)

m_macro_val, m_macro_test = show_model_results(
    'xgboost_macro_only', available_macro,
    train_df, val_df, test_df,
    macro_val_pred, macro_test_pred, TARGET
)
all_metrics.extend([m_macro_val, m_macro_test])

## Model 2 — XGBoost News-Only

In [ ]:
available_news = [f for f in NEWS_FEATURES if f in train_df.columns]

xgb_news = train_xgb(train_df, available_news, TARGET, best_params_news)

news_val_pred = predict_xgb(xgb_news, val_df, available_news)
news_test_pred = predict_xgb(xgb_news, test_df, available_news)

m_news_val, m_news_test = show_model_results(
    'xgboost_news_only', available_news,
    train_df, val_df, test_df,
    news_val_pred, news_test_pred, TARGET
)
all_metrics.extend([m_news_val, m_news_test])

## Model 3 — XGBoost Macro + News

In [ ]:
available_all = [f for f in ALL_FEATURES if f in train_df.columns]

xgb_mn = train_xgb(train_df, available_all, TARGET, best_params_mn)

mn_val_pred = predict_xgb(xgb_mn, val_df, available_all)
mn_test_pred = predict_xgb(xgb_mn, test_df, available_all)

m_mn_val, m_mn_test = show_model_results(
    'xgboost_macro_news', available_all,
    train_df, val_df, test_df,
    mn_val_pred, mn_test_pred, TARGET
)
all_metrics.extend([m_mn_val, m_mn_test])

## Model Comparison — Honest Assessment

In [ ]:
comparison_df = pd.DataFrame(all_metrics)
test_comparison = comparison_df[comparison_df['period'] == 'test'][['model', 'MAE', 'RMSE', 'DA', 'n']]

print('=== TEST SET COMPARISON ===')
print(test_comparison.to_string(index=False))
print()

# Does macro+news beat macro-only?
macro_mae = m_macro_test['MAE']
mn_mae = m_mn_test['MAE']

if mn_mae < macro_mae:
    improvement = (macro_mae - mn_mae) / macro_mae * 100
    print(f'RESULT: Macro+News IMPROVES over Macro-Only by {improvement:.1f}% MAE.')
    print('  → News features provide incremental forecasting value.')
else:
    print(f'WARNING: Macro+News does NOT beat Macro-Only.')
    print(f'  Macro-Only MAE: {macro_mae:.1f}K')
    print(f'  Macro+News MAE: {mn_mae:.1f}K')
    print('  → Do NOT claim news improves forecasting based on this result.')
    print('  → RAG explanation analysis remains valid regardless of this finding.')

## Save models, predictions, and metrics

In [ ]:
# Save models
macro_model_path = MODELS_DIR / 'xgboost_macro_only.joblib'
news_model_path = MODELS_DIR / 'xgboost_news_only.joblib'
mn_model_path = MODELS_DIR / 'xgboost_macro_news.joblib'

joblib.dump(xgb_macro, macro_model_path)
joblib.dump(xgb_news, news_model_path)
joblib.dump(xgb_mn, mn_model_path)

# Save predictions
def save_preds(dates, actuals, preds, model_name, path):
    df = pd.DataFrame({'forecast_date': dates, 'actual': actuals, 'prediction': preds, 'model_name': model_name})
    df.to_parquet(path, index=False)
    return df

macro_pred_df = save_preds(test_df['forecast_date'], test_df[TARGET], macro_test_pred,
                            'xgboost_macro_only', PREDS_DIR / 'predictions_macro_only.parquet')
news_pred_df = save_preds(test_df['forecast_date'], test_df[TARGET], news_test_pred,
                          'xgboost_news_only', PREDS_DIR / 'predictions_news_only.parquet')
mn_pred_df = save_preds(test_df['forecast_date'], test_df[TARGET], mn_test_pred,
                        'xgboost_macro_news', PREDS_DIR / 'predictions_macro_news.parquet')

# Save metrics
comparison_path = METRICS_DIR / 'model_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)

metrics_json = {m['model'] + '_' + m['period']: {k: v for k, v in m.items() if k not in ['model', 'period']}
                for m in all_metrics}
metrics_json_path = METRICS_DIR / 'model_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_json, f, indent=2)

print('Models saved:', macro_model_path.name, news_model_path.name, mn_model_path.name)
print('Predictions saved to:', PREDS_DIR)
print('Metrics saved:', comparison_path)

## Manual Approval Gate

**Before approving:**
1. All 3 models produced non-trivial predictions
2. Comparison table is complete
3. You have read the honest assessment above
4. Predictions for test period look plausible
5. Models beat the naive baseline on at least one metric

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
approval = {
    'approved': True,
    'notebook': '09_train_xgboost_models.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(MODEL_READY_PATH)],
    'output_artifacts': [str(macro_model_path), str(news_model_path), str(mn_model_path),
                         str(comparison_path), str(metrics_json_path)],
    'row_counts': {'test_predictions': int(len(test_df))},
    'warnings': [] if mn_mae < macro_mae else ['macro+news does not beat macro-only'],
    'notes': f'macro_only MAE={m_macro_test["MAE"]}, macro_news MAE={m_mn_test["MAE"]}'
}
ap_path = DRIVE_ROOT / 'approvals' / '09_xgboost_models_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 09 complete. Proceed to 10_model_audit_and_error_analysis.ipynb')